# CATI frozen Mandate003 results
One registered run only. This notebook reads artifacts and never executes the evaluator.
Counterfactuals are not selected portfolio trades; paired families remain runtime-capability blocked.

In [1]:
from pathlib import Path
import json, gzip, hashlib
import pandas as pd
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"docs/research/cati_edge003/results.json").exists())
out=root/"docs/research/cati_edge003"
r=json.loads((out/"results.json").read_text())
assert r["evaluation_runs"]==1 and r["holdout_query_count"]==0
for name,h in r["outcome_population_hashes"].items(): assert hashlib.sha256((out/name).read_bytes()).hexdigest()==h
pd.DataFrame({f:v["pooled"] for f,v in r["families"].items()}).T

Artifact identity/nonoverlap checks passed.


In [2]:
labels=[json.loads(x) for x in gzip.decompress((out/"selected_portfolio_labels.jsonl.gz").read_bytes()).splitlines()]
assert len({x["label_id"] for x in labels})==len(labels)
for family,v in r["families"].items():
    rows=sorted([x for x in labels if x["family"]==family],key=lambda x:x["entry_time"])
    assert len(rows)==v["selected_portfolio_trades"]
    assert all(a["exit_time"]<b["entry_time"] for a,b in zip(rows,rows[1:]))
pd.read_csv(out/"fold_metrics.csv")

Artifact identity/nonoverlap checks passed.


Funding cashflows use historical settled rates and a latest causal hourly mark notional proxy; they are not broker payment receipts.
Basis attribution is inside price return, not an extra return. PRICE_R+BASIS_R+FUNDING_R=GROSS_R for carry.
Leg excursions use native15m bounds solely for diagnostics, never to infer paired basket stop/target touches.